<a href="https://colab.research.google.com/github/yuniecorn-dev/esaa_assignment/blob/main/YB_0410_%EC%97%B0%EC%8A%B5%EB%AC%B8%EC%A0%9C_%EB%B6%84%EB%A5%981.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 결정 트리 훈련

1-a . make_moons (n_sample=1000, noise=0.4)를 사용해 데이터셋을 생성합니다.

In [1]:
from sklearn.datasets import make_moons

X,y = make_moons(n_samples=1000, noise=0.4, random_state=42)

1-b. 이를 train_test_split( )을 사용해 훈련 세트와 테스트 세트로 나눕니다.(random_state=42, train:test = 8:2 비율로)


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

1-c. DecisionTreeClassifier의 최적의 매개변수를 찾기 위해 교차 검증과 함께 그리드 탐색을 수행합니다. (GridSearchCV)

힌트 : max_depth, min_samples_split, max_leaf_nodes 등 하이퍼파라미터를 다양하게 적용해보세요! (교재 pg. 113, 207 참고)

In [17]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

params = {'max_depth': [3,5,7,9], 'min_samples_split' : [2,3,4], 'max_leaf_nodes' : list(range(2,100))}
dtree = DecisionTreeClassifier(random_state=42)

# 1. GridSearchCV 객체 생성
grid_search_cv = GridSearchCV(dtree, params, cv=3, n_jobs=-1)

# 2. 훈련 데이터로 모델 학습시키기 (이때 그리드 탐색이 진행됩니다)
grid_search_cv.fit(X_train, y_train)

GridSearchCV(cv=3, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [3, 5, 7, 9],
                         'max_leaf_nodes': [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12,
                                            13, 14, 15, 16, 17, 18, 19, 20, 21,
                                            22, 23, 24, 25, 26, 27, 28, 29, 30,
                                            31, ...],
                         'min_samples_split': [2, 3, 4]})

In [18]:
# 최적의 매개변수 확인
grid_search_cv.best_estimator_

DecisionTreeClassifier(max_depth=5, max_leaf_nodes=15, random_state=42)

1-d. 찾은 매개변수를 사용해 전체 훈련 세트에 대해 모델을 훈련시키고 테스트 세트에서 성능(정확도)을 측정합니다. (대략 85~87% 정도 나옵니다.)

In [19]:
# GridSearchCV는 최적의 모델로 다시 훈련시키기 때문에 별도의 훈련이 필요없습니다.
from sklearn.metrics import accuracy_score

y_pred = grid_search_cv.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"테스트 세트 정확도: {accuracy:.4f}")

테스트 세트 정확도: 0.5264


# 2. 랜덤포레스트

In [6]:
# 데이터 다운로드 링크로 데이터를 코랩에 불러옵니다.

!wget 'https://bit.ly/3i4n1QB'

import zipfile
with zipfile.ZipFile('3i4n1QB', 'r') as existing_zip:
    existing_zip.extractall('data')

--2026-04-10 09:49:39--  https://bit.ly/3i4n1QB
Resolving bit.ly (bit.ly)... 67.199.248.11, 67.199.248.10
Connecting to bit.ly (bit.ly)|67.199.248.11|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://drive.google.com/uc?export=download&id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw [following]
--2026-04-10 09:49:39--  https://drive.google.com/uc?export=download&id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw
Resolving drive.google.com (drive.google.com)... 142.251.108.139, 142.251.108.100, 142.251.108.138, ...
Connecting to drive.google.com (drive.google.com)|142.251.108.139|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw&export=download [following]
--2026-04-10 09:49:39--  https://drive.usercontent.google.com/download?id=1emLrrpFWT8dCoj5BJb12-5QMG2-nruUw&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)...

In [7]:
# 라이브러리 불러오기

import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

In [9]:
train = pd.read_csv('data/train.csv')

# Scaling
scaler = MinMaxScaler()
scaler.fit(train[['fixed acidity']])
train['Scaled fixed acidity'] = scaler.transform(train[['fixed acidity']])

# Encoding
encoder = OneHotEncoder()
encoder.fit(train[['type']])
onehot = encoder.transform(train[['type']])
onehot = onehot.toarray()
onehot = pd.DataFrame(onehot)
onehot.columns = encoder.get_feature_names_out()
train = pd.concat([train, onehot], axis = 1)
train = train.drop(columns = ['type'])
train.head()

,index,quality,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,Scaled fixed acidity,type_red,type_white
0,0,5,5.6,0.695,0.06,6.8,0.042,9.0,84.0,0.99432,3.44,0.44,10.2,0.148760,0.0,1.0
1,1,5,8.8,0.610,0.14,2.4,0.067,10.0,42.0,0.99690,3.19,0.59,9.5,0.413223,1.0,0.0
2,2,5,7.9,0.210,0.39,2.0,0.057,21.0,138.0,0.99176,3.05,0.52,10.9,0.338843,0.0,1.0
3,3,6,7.0,0.210,0.31,6.0,0.046,29.0,108.0,0.99390,3.26,0.50,10.8,0.264463,0.0,1.0
4,4,6,7.8,0.400,0.26,9.5,0.059,32.0,178.0,0.99550,3.04,0.43,10.9,0.330579,0.0,1.0


2-a. 랜덤포레스트 분류 모형을 불러옵니다.

In [10]:
from sklearn.ensemble import RandomForestClassifier

2-b. 랜덤포레스트 분류 모형을 "random_classifier"라는 변수에 저장합니다.

In [11]:
random_classifier = RandomForestClassifier(random_state=42)

2-c. "X"라는 변수에 train의 "quality" 피쳐를 제거하고 저장합니다.

In [12]:
X = train.drop(columns=['quality'])

2-d. "y"라는 변수에 정답인 train의 "quality" 피쳐를 저장합니다.

In [13]:
y = train['quality']

2-e. "X"와 "y"를 train set과 test set으로 분할합니다. (test_size=0.2)

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

2-f. "random_classifier"를 X_train과 y_train을 이용해 학습시켜보세요. 이후, 학습시킨 모델로 X_test 값을 통해 y_pred 값을 예측해보세요.

In [15]:
# 모델 학습
random_classifier.fit(X_train, y_train)

# 예측 수행
y_pred = random_classifier.predict(X_test)

2-g. 예측된 결과를 confusion matrix와 Accuracy를 이용해 평가해보세요.

In [16]:
from sklearn.metrics import confusion_matrix, accuracy_score

# 오차 행렬 (Confusion Matrix) 출력
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 정확도 (Accuracy) 출력
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")

Confusion Matrix:
[[  0   0   1   2   0   0   0]
 [  0   3  18  14   0   0   0]
 [  0   1 246  99   5   0   0]
 [  0   0  77 387  29   0   0]
 [  0   0   3  84  95   1   0]
 [  0   0   0  14   8  12   0]
 [  0   0   0   1   0   0   0]]

Accuracy: 0.6755
